
# Scientific programming video game prediction

## Library justification
The first step of running this code would be to import the packages used throughout the script:
- For data handling and preprocessing the main packages that are important are `pandas` for data manipulation and `parallel_coordinates` for visualizing multivariate patterns. `numpy` provides optimized numerical operations and array handling.
- `scikit-learn` is the core library used for all machine learning throughout the script. The package is also used for importing the classification models.
- For multivariate outlier detection, `pyod.modles.KNN` and `pyod.models.IForest` are used. 
- `matplotlib.pyplot` and `seaborn` are used to create visualizations. `scipy.cluster.hierarchy` is used to perform hierarchical clustering and dendogram visualizations.
- `tqdm` and `joblib` are used to add a progress bar and to enable parallel computation to speed up certain processes.

In [ ]:
# Execute this cell to import the required packages
import pandas as pd
from pandas.plotting import parallel_coordinates
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_validate, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from pyod.models.knn import KNN
from pyod.models.iforest import IForest
from tqdm import tqdm
from joblib import Parallel, delayed

# Loading the dataset + descriptive stats + imputation
- This block of code loads the data as well as the clean version of the data. 
- It checks whether there is missingness in the data or impossible values.
- Boxplots are created for every numerical column in the dataset to see the distributions. 
- An iterative imputer using random forest is created and used to impute the nan values in the data.
- Unsupervised PCA, t-SNE and K-means clustering plots are made to visualize the data
- Without parallel computation, imputation took ~15 min and now it is reduced to only ~2 mins.

In [ ]:
# Load the videogame data
data = pd.read_csv("Videogamedata_dirty.csv", sep=";")
cleandata = pd.read_csv("Videogamedata.csv")

# Check for duplicates
num_dupes = data['gameId'].duplicated().sum()
print("Number of duplicate game IDs:", num_dupes)

# Remove the row containing the impossible value
data = data[data['blueHeralds'] >= 0]

# Check for missing values
print(data.isnull().sum().sort_values(ascending=False))

# Distribution visualizations
numeric_cols = [col for col in data.select_dtypes(include=['number']).columns if col != 'gameId']

for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=data[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

# Imputation (dirty data version)
X = data[numeric_cols]

# Parallelized Iterative Imputer
imp = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # enables parallel computation in RF
    ),
    max_iter=3,
    random_state=42
)

X_imputed = imp.fit_transform(X)

imputed_df_dirty = pd.DataFrame(X_imputed, columns=numeric_cols, index=data.index)

## Unsupervised visualization section
# Standardize the data for fair comparison
scaler = StandardScaler()
X_scaled = scaler.fit_transform(imputed_df_dirty)

# PCA for variance and clustering
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)
imputed_df_dirty["PCA1"] = pca_result[:, 0]
imputed_df_dirty["PCA2"] = pca_result[:, 1]

# Define a color-blind-friendly palette (Okabe–Ito)
cb_palette = ["#0072B2", "#E69F00", "#009E73", "#56B4E9", "#D55E00", "#CC79A7"]

# PCA visualization
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=imputed_df_dirty,
    x="PCA1",
    y="PCA2",
    hue="blueWins",
    palette=cb_palette[:2],
    s=40,
    alpha=0.7
)
plt.title("PCA projection of imputed (dirty) data")
plt.xlabel(f"PCA1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PCA2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.legend(title="Blue Wins")  # Let Seaborn handle labels automatically
plt.show()

# t-SNE visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate=200)
tsne_result = tsne.fit_transform(X_scaled)
imputed_df_dirty["tSNE1"] = tsne_result[:, 0]
imputed_df_dirty["tSNE2"] = tsne_result[:, 1]

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=imputed_df_dirty,
    x="tSNE1",
    y="tSNE2",
    hue="blueWins",
    palette=cb_palette[:2],
    s=40,
    alpha=0.7
)
plt.title("t-SNE visualization of data structure (colored by blueWins)")
plt.legend(title="Blue Wins")  # Again, Seaborn handles label-color mapping
plt.show()

# K-Means clustering visualization 
kmeans = KMeans(n_clusters=3, random_state=42)
imputed_df_dirty["Cluster"] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=imputed_df_dirty,
    x="PCA1",
    y="PCA2",
    hue="Cluster",
    palette=cb_palette[:3],
    s=50,
    alpha=0.8
)
plt.title("K-Means clustering visualized on PCA projection")
plt.legend(title="Cluster")
plt.show()

# Testing the parameter settings of the randomforest imputation
- The code below tests how the iterative imputer (with a Random Forest regressor as estimator) converges when filling in missing values:
- Only numeric columns (excluding identifiers like `gameId`) are kept for imputation.
- An IterativeImputer is configured to predict and fill missing values using a random forest regressor.
- For each iteration (1 → max_iter), a separate imputer is fit, and the resulting imputed dataset is stored.
- The mean absolute difference between successive imputed datasets is computed to see how much the imputations are changing.
- The mean absolute change per iteration is plotted. A decreasing curve indicates the imputer is stabilizing.
- If the mean change drops below the threshold (`1e-3`), a message is printed to indicate convergence.
- The code below took ~400 minutes to run without implementing parallel computing, with parallel computing it takes ~45 minutes

In [ ]:
# Use only the numerical columns
numeric_cols = [col for col in data.select_dtypes(include=['number']).columns if col != 'gameId']
X = data[numeric_cols].copy()

# Define max iterations and number of CPU cores
MAX_ITER = 10
N_CORES = -1  # use all available CPU cores (same as in codeblock above)

# Define one iteration of imputation
def run_imputation(iters):
    imp_iter = IterativeImputer(
        estimator=RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=N_CORES  # parallelize Random Forest
        ),
        max_iter=iters,
        random_state=42,
        imputation_order="ascending",
        sample_posterior=False
    )
    X_imp = imp_iter.fit_transform(X)
    return X_imp

# Run imputations in parallel across all iterations
imputed_versions = Parallel(n_jobs=N_CORES, prefer="processes", verbose=10)(
    delayed(run_imputation)(iters) for iters in range(1, MAX_ITER + 1)
)

# Compute the mean absolute change between successive iterations 
mean_abs_changes = [
    np.mean(np.abs(imputed_versions[k] - imputed_versions[k - 1]))
    for k in range(1, len(imputed_versions))
]

# Plot the convergence curve
plt.figure(figsize=(6, 4))
plt.plot(range(2, MAX_ITER + 1), mean_abs_changes, marker='o')
plt.xlabel("Iteration")
plt.ylabel("Mean absolute change vs. previous iteration")
plt.title("Convergence of Iterative Imputation (Parallelized)")
plt.grid(True)
plt.show()

# Print when change falls below a threshold
threshold = 1e-3
for i, change in enumerate(mean_abs_changes, start=2):
    if change < threshold:
        print(f"Converged (mean abs. change < {threshold}) at iteration {i}")
        break

In [ ]:
# Prepare data
numeric_cols = [col for col in data.select_dtypes(include=['number']).columns if col != 'gameId']
X = data[numeric_cols].copy()

# Define max iterations, number of estimators, and CPU cores
max_iter = 5
n_estimators_list = [50, 100, 500]
N_CORES = -1  # use all available cores

# Function for one n_estimators setting
def impute_and_track(n_estimators, max_iter, X):
    imputed_versions = []
    for iters in range(1, max_iter + 1):
        imp_iter = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=n_estimators,
                random_state=42,
                n_jobs=N_CORES  # parallel inside RF
            ),
            max_iter=iters,
            random_state=42,
            imputation_order="ascending",
            sample_posterior=False
        )
        X_imp = imp_iter.fit_transform(X)
        imputed_versions.append(X_imp)

    mean_abs_changes = [
        np.mean(np.abs(imputed_versions[k] - imputed_versions[k - 1]))
        for k in range(1, len(imputed_versions))
    ]
    return n_estimators, mean_abs_changes

# Run in parallel again
results = []
with tqdm(total=len(n_estimators_list), desc="Imputing (parallel)") as pbar:
    parallel_results = Parallel(n_jobs=N_CORES, prefer="processes")(
        delayed(impute_and_track)(n, max_iter, X)
        for n in n_estimators_list
    )
    pbar.update(len(n_estimators_list))

# Collect the results
convergence_curves = dict(parallel_results)

# Plot the results
line_styles = ['-', '--', ':']
colors = plt.cm.tab10.colors

plt.figure(figsize=(8, 5))
for idx, (n_estimators, mean_abs_changes) in enumerate(convergence_curves.items()):
    plt.plot(
        range(2, max_iter + 1),
        mean_abs_changes,
        marker='o',
        color=colors[idx % len(colors)],
        linestyle=line_styles[idx % len(line_styles)],
        label=f"{n_estimators} estimators"
    )

plt.xlabel("Iteration")
plt.ylabel("Mean absolute change vs. previous iteration")
plt.title("Convergence of Iterative Imputation (Parallelized)")
plt.grid(True)
plt.legend()
plt.show()

# Calculate the convergence
threshold = 1e-3
for n_estimators, mean_abs_changes in convergence_curves.items():
    for i, change in enumerate(mean_abs_changes, start=2):
        if change < threshold:
            print(f"{n_estimators} estimators converged (mean abs. change < {threshold}) at iteration {i}")
            break